# DuckLake Conference Live Demo

Assumes the table already exists. Run `streaming_demo` first to populate it with
millions of rows. This notebook walks through: connect + explore, ACID transactions,
time travel, schema evolution + CDC, MERGE/upsert, and maintenance.

In [ ]:
import sys
from pathlib import Path

from loguru import logger

logger.remove()
logger.add(sys.stderr, level="INFO")

_REPO = Path.cwd()
if _REPO.name == "notebooks":
    _REPO = _REPO.parent
if str(_REPO / "src") not in sys.path:
    sys.path.insert(0, str(_REPO / "src"))

from ducklake_playground import DuckLakeEngine, load_config

config = load_config(_REPO / "config.yaml")
engine = DuckLakeEngine()
# Must match the storage backend used when `streaming_demo` created the table.
storage_mode = "local"  # or "s3"
engine.setup(config, storage_mode)
con = engine.connection
catalog = engine.catalog_name
TABLE = "demo_table"
fq = f"{catalog}.main.{TABLE}"
print(f"Connected to {catalog} | table: {fq}")

## 1. Explore the Table

In [38]:
con.execute(f"DESCRIBE {fq}").pl()

column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""id""","""BIGINT""","""YES""",null,"""NULL""",null
"""event_date""","""DATE""","""YES""",null,"""NULL""",null
"""int8_col""","""TINYINT""","""YES""",null,"""NULL""",null
"""int16_col""","""SMALLINT""","""YES""",null,"""NULL""",null
"""int32_col""","""INTEGER""","""YES""",null,"""NULL""",null
…,…,…,…,…,…
"""text_col""","""VARCHAR""","""YES""",null,"""NULL""",null
"""bool_col""","""BOOLEAN""","""YES""",null,"""NULL""",null
"""list_col""","""INTEGER[]""","""YES""",null,null,null


In [39]:
con.execute(f"""
    SELECT COUNT(*)                   AS total_rows,
           MIN(event_date)            AS first_date,
           MAX(event_date)            AS last_date,
           COUNT(DISTINCT event_date) AS partitions
    FROM {fq}
""").pl()

total_rows,first_date,last_date,partitions
i64,date,date,i64
1000000,2024-01-01,2024-01-30,30


In [40]:
con.execute(f"""
    SELECT varchar_col,
           COUNT(*)         AS cnt,
           SUM(int64_col)   AS total,
           AVG(float64_col) AS avg_val
    FROM {fq}
    WHERE event_date BETWEEN DATE '2024-01-10' AND DATE '2024-01-15'
    GROUP BY varchar_col
    ORDER BY cnt DESC
    LIMIT 10
""").pl()

varchar_col,cnt,total,avg_val
str,i64,"decimal[38,0]",f64
"""value_529""",249,-151148723910259724750,-1.9045e13
"""value_305""",243,-30540907389986659864,9.2362e13
"""value_922""",239,85405002404505944155,5.4239e12
"""value_800""",239,63740955580357578695,-1.4480e13
"""value_686""",238,58514410144643346574,-1.5752e13
"""value_174""",234,-82304733355348137510,-6.4248e12
"""value_313""",234,-32816697197540210426,-1.0947e13
"""value_818""",233,-97668297955158875317,4.6219e13
"""value_419""",232,-13725971426480809215,-2.7866e13


In [41]:
# Prove partition pruning to the audience
print(con.execute(f"""
    EXPLAIN ANALYZE
    SELECT varchar_col, COUNT(*) AS cnt
    FROM {fq}
    WHERE event_date = DATE '2024-01-15'
    GROUP BY varchar_col
""").pl()["explain_value"].first())

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
     EXPLAIN ANALYZE     SELECT varchar_col, COUNT(*) AS cnt     FROM playground_ducklake_s3.main.demo_table     WHERE event_date = DATE '2024-01-15'     GROUP BY varchar_col 
┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││         HTTPFS HTTP Stats         ││
││                                   ││
││            in: 0 bytes            ││
││            out: 0 bytes           ││
││              #HEAD: 0             ││
││              #GET: 0              ││
││              #PUT: 0              ││
││              #POST: 0             ││
││             #DELETE: 0            ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││

## 2. ACID Transactions

Multi-statement transaction: INSERT + UPDATE land atomically in one DuckLake snapshot.

In [42]:
pre_count = con.execute(f"SELECT COUNT(*) FROM {fq}").fetchone()[0]
pre_snapshot = con.execute(
    f"SELECT MAX(snapshot_id) FROM {catalog}.snapshots()"
).fetchone()[0]
print(f"Before transaction: {pre_count:,} rows | snapshot v{pre_snapshot}")

Before transaction: 1,000,000 rows | snapshot v26


In [43]:
con.execute("BEGIN TRANSACTION")
try:
    # Insert two new rows
    con.execute(f"""
        INSERT INTO {fq} (id, event_date, int64_col, float64_col, varchar_col)
        VALUES
            (900000001, DATE '2024-01-15', 1499, 99.95, 'value_042'),
            (900000002, DATE '2024-01-15', 49,   19.99, 'value_007')
    """)
    # Update one of them (10% discount)
    con.execute(f"""
        UPDATE {fq}
        SET float64_col = float64_col * 0.9
        WHERE id = 900000001
    """)
    con.execute("COMMIT")
    print("COMMITTED")
except Exception as exc:
    con.execute("ROLLBACK")
    print(f"ROLLED BACK: {exc}")

post_count = con.execute(f"SELECT COUNT(*) FROM {fq}").fetchone()[0]
post_snapshot = con.execute(
    f"SELECT MAX(snapshot_id) FROM {catalog}.snapshots()"
).fetchone()[0]
print(f"After transaction: {post_count:,} rows (v{post_snapshot}, +{post_count - pre_count})")

COMMITTED
After transaction: 1,000,002 rows (v27, +2)


In [44]:
# Verify: both the INSERT and UPDATE landed
con.execute(f"""
    SELECT id, event_date, int64_col, float64_col, varchar_col
    FROM {fq}
    WHERE id IN (900000001, 900000002)
    ORDER BY id
""").pl()

id,event_date,int64_col,float64_col,varchar_col
i64,date,i64,f64,str
900000001,2024-01-15,1499,89.955,"""value_042"""
900000002,2024-01-15,49,19.99,"""value_007"""


## 3. Time Travel & Snapshots

Each write creates a new immutable snapshot. Query any historical version.

In [45]:
con.execute(f"""
    SELECT snapshot_id, snapshot_time, changes
    FROM {catalog}.snapshots()
    ORDER BY snapshot_id DESC
    LIMIT 10
""").pl()

snapshot_id,snapshot_time,changes
i64,"datetime[μs, Europe/Rome]",list[struct[2]]
27,2026-05-09 18:40:30.168753 CEST,"[{""inlined_insert"",[""3""]}]"
26,2026-05-09 18:40:14.618219 CEST,"[{""inlined_delete"",[""3""]}]"
25,2026-05-09 18:39:33.575059 CEST,"[{""inlined_insert"",[""3""]}, {""inlined_delete"",[""3""]}]"
24,2026-05-09 18:39:30.914870 CEST,"[{""tables_altered"",[""3""]}]"
23,2026-05-09 18:39:29.697708 CEST,"[{""tables_altered"",[""3""]}]"
22,2026-05-09 18:39:26.512349 CEST,"[{""tables_altered"",[""3""]}]"
21,2026-05-09 18:39:20.450917 CEST,"[{""tables_altered"",[""3""]}]"
20,2026-05-09 18:39:18.419950 CEST,"[{""tables_altered"",[""3""]}]"
19,2026-05-09 18:38:59.771112 CEST,"[{""tables_altered"",[""3""]}]"


In [46]:
print(f"Before-tx snapshot: v{pre_snapshot} | After-tx snapshot: v{post_snapshot}")

Before-tx snapshot: v26 | After-tx snapshot: v27


In [47]:
# Query the table AS OF the previous snapshot (before our transaction)
con.execute(f"""
    SELECT COUNT(*) AS row_count_before_tx
    FROM {fq} AT (VERSION => {pre_snapshot})
""").pl()

row_count_before_tx
i64
1000000


In [48]:
# Prove the inserted rows did NOT exist in the previous version
con.execute(f"""
    SELECT id, event_date, float64_col
    FROM {fq} AT (VERSION => {pre_snapshot})
    WHERE id IN (900000001, 900000002)
""").pl()

id,event_date,float64_col
i64,date,f64


## 4. Change Data Feed (CDC)

What changed between two snapshots? No Kafka or external tooling required.

In [49]:
con.execute(f"""
    SELECT *
    FROM {catalog}.table_changes('{TABLE}', {pre_snapshot}, {post_snapshot})
    ORDER BY change_type, id
    LIMIT 20
""").pl()

snapshot_id,rowid,change_type,id,event_date,int8_col,int16_col,int32_col,int64_col,float32_col,float64_col,decimal_col,datetime_col,timestamp_col,varchar_col,text_col,bool_col,list_col,struct_col,map_col
i64,i64,str,i64,date,i8,i16,i32,i64,f32,f64,"decimal[18,4]",datetime[μs],"datetime[μs, Europe/Rome]",str,str,bool,list[i32],struct[3],list[struct[2]]
26,1000001,"""delete""",900000001,2024-01-15,null,null,null,9999,null,42.0,null,null,null,"""value_042""",null,null,null,null,[]
26,1000006,"""delete""",900000001,2024-01-15,null,null,null,9999,null,42.0,null,null,null,"""value_042""",null,null,null,null,[]
26,1000004,"""delete""",900000001,2024-01-15,null,null,null,9999,null,42.0,null,null,null,"""value_042""",null,null,null,null,[]
26,1000005,"""delete""",900000002,2024-01-15,null,null,null,49,null,19.99,null,null,null,"""value_007""",null,null,null,null,[]
26,1000003,"""delete""",900000002,2024-01-15,null,null,null,49,null,19.99,null,null,null,"""value_007""",null,null,null,null,[]
26,1000000,"""delete""",900000002,2024-01-15,null,null,null,49,null,19.99,null,null,null,"""value_007""",null,null,null,null,[]
26,1000002,"""delete""",999999999,2024-01-20,null,null,null,7777,null,55.5,null,null,null,"""value_001""",null,null,null,null,[]
27,1000008,"""insert""",900000001,2024-01-15,null,null,null,1499,null,89.955,null,null,null,"""value_042""",null,null,null,null,[]
27,1000007,"""insert""",900000002,2024-01-15,null,null,null,49,null,19.99,null,null,null,"""value_007""",null,null,null,null,[]


## 5. Schema Evolution

ADD / RENAME / DROP columns without rewriting any Parquet files.

In [50]:
# Add a new column (metadata-only, no Parquet rewrite)
con.execute(f"ALTER TABLE {fq} ADD COLUMN priority VARCHAR DEFAULT 'normal'")
print("Column 'priority' added. Existing Parquet files untouched.")

Column 'priority' added. Existing Parquet files untouched.


In [51]:
# Verify: new column appears, old rows have the default
con.execute(f"""
    SELECT id, event_date, varchar_col, priority
    FROM {fq}
    WHERE id IN (900000001, 900000002, 1, 2, 3)
    ORDER BY id
    LIMIT 5
""").pl()

id,event_date,varchar_col,priority
i64,date,str,str
1,2024-01-01,"""value_270""","""normal"""
2,2024-01-01,"""value_947""","""normal"""
3,2024-01-01,"""value_260""","""normal"""
900000001,2024-01-15,"""value_042""","""normal"""
900000002,2024-01-15,"""value_007""","""normal"""


In [52]:
# Rename column (metadata-only, zero Parquet I/O)
con.execute(f"ALTER TABLE {fq} RENAME COLUMN priority TO urgency")
print("Renamed 'priority' to 'urgency'. Zero file I/O.")

Renamed 'priority' to 'urgency'. Zero file I/O.


In [53]:
# Drop column to restore the table for the next demo
con.execute(f"ALTER TABLE {fq} DROP COLUMN urgency")
print("Dropped 'urgency'. Schema restored.")

Dropped 'urgency'. Schema restored.


## 6. MERGE / Upsert

Atomic upsert: update existing rows + insert new ones in a single snapshot.

In [54]:
con.execute(f"""
    MERGE INTO {fq} AS target
    USING (
        VALUES
            (900000001, DATE '2024-01-15', CAST(9999 AS BIGINT),
             CAST(42.0 AS DOUBLE), 'value_042'),
            (999999999, DATE '2024-01-20', CAST(7777 AS BIGINT),
             CAST(55.5 AS DOUBLE), 'value_001')
    ) AS source(id, event_date, int64_col, float64_col, varchar_col)
    ON target.id = source.id
    WHEN MATCHED THEN
        UPDATE SET int64_col = source.int64_col,
                   float64_col = source.float64_col
    WHEN NOT MATCHED THEN
        INSERT (id, event_date, int64_col, float64_col, varchar_col)
        VALUES (source.id, source.event_date, source.int64_col,
                source.float64_col, source.varchar_col)
""")
print("MERGE complete: id=900000001 updated, id=999999999 inserted.")

MERGE complete: id=900000001 updated, id=999999999 inserted.


In [55]:
# Verify MERGE results
con.execute(f"""
    SELECT id, event_date, int64_col, float64_col, varchar_col
    FROM {fq}
    WHERE id IN (900000001, 999999999)
    ORDER BY id
""").pl()

id,event_date,int64_col,float64_col,varchar_col
i64,date,i64,f64,str
900000001,2024-01-15,9999,42.0,"""value_042"""
999999999,2024-01-20,7777,55.5,"""value_001"""


## 7. Maintenance

DuckLake **does** require maintenance: file compaction, snapshot expiry, and cleanup.

In [56]:
# File statistics BEFORE compaction
con.execute(f"""
    SELECT COUNT(*)                                    AS total_files,
           ROUND(SUM(data_file_size_bytes) / 1e6, 2)  AS total_mb,
           ROUND(AVG(data_file_size_bytes) / 1e6, 2)  AS avg_file_mb,
           ROUND(MIN(data_file_size_bytes) / 1e6, 2)  AS min_file_mb,
           ROUND(MAX(data_file_size_bytes) / 1e6, 2)  AS max_file_mb
    FROM ducklake_list_files('{catalog}', '{TABLE}')
""").pl()

total_files,total_mb,avg_file_mb,min_file_mb,max_file_mb
i64,f64,f64,f64,f64
32,170.84,5.34,0.0,5.7


In [57]:
# Compact small files into larger ones
con.execute(f"CALL ducklake_merge_adjacent_files('{catalog}')")
print("ducklake_merge_adjacent_files complete.")

ducklake_merge_adjacent_files complete.


In [58]:
# File statistics AFTER compaction
con.execute(f"""
    SELECT COUNT(*)                                    AS total_files,
           ROUND(SUM(data_file_size_bytes) / 1e6, 2)  AS total_mb,
           ROUND(AVG(data_file_size_bytes) / 1e6, 2)  AS avg_file_mb
    FROM ducklake_list_files('{catalog}', '{TABLE}')
""").pl()

total_files,total_mb,avg_file_mb
i64,f64,f64
32,170.84,5.34


In [59]:
# Expire old snapshots (aggressive for demo: 1 day)
con.execute(f"CALL ducklake_expire_snapshots('{catalog}', older_than => now() - INTERVAL '1 day')")
print("ducklake_expire_snapshots complete.")

# Clean up orphaned data files
con.execute(f"CALL ducklake_cleanup_old_files('{catalog}', cleanup_all => true)")
print("ducklake_cleanup_old_files complete.")

ducklake_expire_snapshots complete.
ducklake_cleanup_old_files complete.


In [ ]:
# Final snapshot list
con.execute(f"""
    SELECT snapshot_id, snapshot_time, changes
    FROM {catalog}.snapshots()
    ORDER BY snapshot_id DESC
    LIMIT 10
""").pl()

snapshot_id,snapshot_time,changes
i64,"datetime[μs, Europe/Rome]",list[struct[2]]
31,2026-05-09 18:40:40.124998 CEST,"[{""tables_inserted_into"",[""3""]}, {""inlined_insert"",[""3""]}, {""inlined_delete"",[""3""]}]"
30,2026-05-09 18:40:38.944936 CEST,"[{""tables_altered"",[""3""]}]"
29,2026-05-09 18:40:38.555364 CEST,"[{""tables_altered"",[""3""]}]"
28,2026-05-09 18:40:35.800480 CEST,"[{""tables_altered"",[""3""]}]"
27,2026-05-09 18:40:30.168753 CEST,"[{""inlined_insert"",[""3""]}]"
26,2026-05-09 18:40:14.618219 CEST,"[{""inlined_delete"",[""3""]}]"
25,2026-05-09 18:39:33.575059 CEST,"[{""inlined_insert"",[""3""]}, {""inlined_delete"",[""3""]}]"
24,2026-05-09 18:39:30.914870 CEST,"[{""tables_altered"",[""3""]}]"
23,2026-05-09 18:39:29.697708 CEST,"[{""tables_altered"",[""3""]}]"


## Cleanup

Remove demo artifacts to restore the table to its pre-demo state.

In [36]:
# Uncomment and run after the presentation:
con.execute(f"DELETE FROM {fq} WHERE id IN (900000001, 900000002, 999999999)").pl()
# print("Demo rows removed.")

Count
i64
0


In [ ]:
# engine.close()